## `pybiglasso` Tuning: Columnar Disk Formats

biglasso needs to be able to work with larger-than-memory datasets.

Here we compare Zarr, Parquet, and numpy.memmap in terms of read speed and basic compute.

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [139]:
import tempfile

import random

import numpy as np

import zarr
from zarr import Zlib, BZ2, LZMA, Blosc
#from zarr.blosc import NOSHUFFLE, BITSHUFFLE
from numcodecs import blosc

import pyarrow as pa
import pyarrow.parquet as pq

from tqdm import tqdm

import ctypes
import mmap

In [60]:
def simulate_genotype_data(nrow, ncol, dosage_type="genotyped", dtype="f8", seed=1234):
    
    np.random.seed(seed)

    af = np.random.beta(0.3, 0.3, size=ncol)
    dosages = np.random.binomial(2, af, size=(nrow, ncol)).astype(dtype)
    
    if dosage_type == "imputed":
        dosages += np.random.normal(loc=0, scale=np.minimum(af, 1-af) / 100, size=(nrow, ncol))
        dosages = dosages.clip(0, 2)

    return dosages

simulate_genotype_data(20, 3)
simulate_genotype_data(20, 3, dosage_type="imputed")

array([[0., 1., 1.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 1., 2.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 1., 2.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 0., 2.],
       [0., 1., 2.],
       [1., 1., 2.],
       [0., 0., 2.],
       [0., 0., 2.]])

array([[6.83980776e-05, 9.99955708e-01, 1.00016514e+00],
       [2.98627173e-04, 0.00000000e+00, 1.99997947e+00],
       [5.95007567e-05, 0.00000000e+00, 2.00000000e+00],
       [0.00000000e+00, 2.53255769e-03, 1.99966646e+00],
       [4.09366780e-05, 1.00087892e+00, 1.99977073e+00],
       [8.92717025e-05, 8.78309578e-04, 2.00000000e+00],
       [0.00000000e+00, 2.50417071e-03, 2.00000000e+00],
       [0.00000000e+00, 7.88201296e-04, 2.00000000e+00],
       [8.97223674e-05, 0.00000000e+00, 2.00000000e+00],
       [2.94888045e-05, 2.05200343e-04, 1.99987445e+00],
       [1.48258565e-04, 1.00122839e+00, 2.00000000e+00],
       [2.68937039e-04, 9.95788946e-05, 1.99988325e+00],
       [0.00000000e+00, 0.00000000e+00, 2.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 2.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 2.00000000e+00],
       [1.09456470e-04, 6.28076878e-04, 2.00000000e+00],
       [9.35663094e-05, 1.00170053e+00, 1.99977199e+00],
       [9.99909577e-01, 1.00152

### Creating test data sets

In [161]:
nrow = 1_000_000
ncol = 5000
dtype = "f8"

blocksize = 25
nblocks = ncol // blocksize

In [150]:
%pdb

Automatic pdb calling has been turned OFF


In [162]:
#Xz = zarr.open('example.zarr', mode='w', fill_value=None, shape=(nrow, ncol), chunks=(nrow, None), dtype=dtype, order="F")

!rm -rf example.zarr
Xz = zarr.empty(shape=(nrow, ncol), chunks=(nrow, 1), dtype=dtype, order="F",
                store="example.zarr",
               compressor=Blosc(cname='lz4', clevel=3, shuffle=Blosc.NOSHUFFLE, blocksize=0)
               )
Xz.info

Xmm = np.lib.format.open_memmap("example.mmap", mode="w+", dtype=dtype, shape=(nrow, ncol), fortran_order=True)


for ix in tqdm(range(nblocks)):
    coldata = simulate_genotype_data(nrow, blocksize, dosage_type="genotyped", dtype=dtype)
    Xz[:, ix * blocksize : (ix + 1) * blocksize] = coldata[:, :]
    Xmm[:, ix * blocksize : (ix + 1) * blocksize] = coldata[:, :]

del Xz
del Xmm

Type,zarr.core.Array
Data type,float64
Shape,"(1000000, 5000)"
Chunk shape,"(1000000, 1)"
Order,F
Read-only,False
Compressor,"Blosc(cname='lz4', clevel=3, shuffle=NOSHUFFLE, blocksize=0)"
Store type,zarr.storage.DirectoryStore
No. bytes,40000000000 (37.3G)
No. bytes stored,349
Storage ratio,114613180.5


100%|██████████| 200/200 [06:48<00:00,  2.04s/it]


In [166]:
Xz = zarr.open('example.zarr', mode='r')
Xmm = np.lib.format.open_memmap("example.mmap", mode="r")
    
Xz.info
Xmm.shape

Xz[:, 10].sum()
Xmm[:, 10].sum()

Type,zarr.core.Array
Data type,float64
Shape,"(1000000, 5000)"
Chunk shape,"(1000000, 1)"
Order,F
Read-only,True
Compressor,"Blosc(cname='lz4', clevel=3, shuffle=NOSHUFFLE, blocksize=0)"
Store type,zarr.storage.DirectoryStore
No. bytes,40000000000 (37.3G)
No. bytes stored,3947965349 (3.7G)
Storage ratio,10.1


(1000000, 5000)

185523.0

185523.0

In [164]:
# -- read a random column

def random_column_sum(X):
    col_ix = random.randint(0, X.shape[1] - 1)
    return X[:, col_ix].sum()

random_column_sum(Xz)
random_column_sum(Xmm)


%timeit random_column_sum(Xz)
%timeit random_column_sum(Xmm)

Xmm._mmap.madvise(mmap.MADV_SEQUENTIAL)
%timeit random_column_sum(Xmm)

275.0

10006.0

5.7 ms ± 169 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
3.13 ms ± 283 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.68 ms ± 455 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [169]:
# -- do a full sweep

def full_sweep(X):
    total = 0.0
    for ix in range(X.shape[1]):
        total += X[:, ix].sum()
    return total

full_sweep(Xz)
full_sweep(Xmm)


%timeit full_sweep(Xz)
%timeit full_sweep(Xmm)

Xmm._mmap.madvise(mmap.MADV_SEQUENTIAL)
%timeit full_sweep(Xmm)

3180394600.0

3180394600.0

26.4 s ± 1.8 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
24 s ± 820 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
30.1 s ± 946 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [170]:
Xzm = zarr.empty(shape=(nrow, ncol), chunks=(nrow, 1), dtype=dtype, order="F",
                #store="example.zarr",
               compressor=Blosc(cname='lz4', clevel=3, shuffle=Blosc.NOSHUFFLE, blocksize=0)
               )
Xzm.info

for ix in tqdm(range(Xzm.shape[1])):
    Xzm[:, ix] = Xz[:, ix]

Xzm.info

random_column_sum(Xzm)

%timeit random_column_sum(Xzm)

full_sweep(Xzm)

%timeit full_sweep(Xzm)

Type,zarr.core.Array
Data type,float64
Shape,"(1000000, 5000)"
Chunk shape,"(1000000, 1)"
Order,F
Read-only,False
Compressor,"Blosc(cname='lz4', clevel=3, shuffle=NOSHUFFLE, blocksize=0)"
Store type,builtins.dict
No. bytes,40000000000 (37.3G)
No. bytes stored,349
Storage ratio,114613180.5


100%|██████████| 5000/5000 [00:58<00:00, 85.22it/s]


Type,zarr.core.Array
Data type,float64
Shape,"(1000000, 5000)"
Chunk shape,"(1000000, 1)"
Order,F
Read-only,False
Compressor,"Blosc(cname='lz4', clevel=3, shuffle=NOSHUFFLE, blocksize=0)"
Store type,builtins.dict
No. bytes,40000000000 (37.3G)
No. bytes stored,3947965349 (3.7G)
Storage ratio,10.1


16064.0

4.73 ms ± 171 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


3180394600.0

23.7 s ± 27.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
